# 📊 SOS + IL — Pipeline Completo de Métricas
### Strength of Schedule · Z-Score Ajustado · Custo/Valor do Gol · Índice de Legitimidade

---

## O que este notebook faz (em ordem)

| Etapa | Métricas geradas |
|---|---|
| **1 — SOS** | `sos_home/away_raw/norm/n` — força da tabela via ELO ponderado |
| **2 — Z-Score** | `z_gols/xg/vit_home/away` + `_adj` — desvio da média ajustado pelo SOS |
| **3 — Custo/Valor do Gol** | `custo/valor_gol_casa/fora` — lógica Adriano Duarte com gols ajustados por xG |
| **4 — Luck Factor** | `luck_home/away` — ΔxG (gols − xG esperado) |
| **5 — Delta** | `delta_bc/bf` — divergência modelo vs mercado |
| **6 — IL** | `IL_casa/fora` + `classificacao_IL` — índice 0-100 de legitimidade |

---

## Equações centrais

```
SOS_raw      = Σ(ELO_adv_i × decay^i) / Σ(decay^i)      [últimos N jogos]
SOS_norm     = SOS_raw / ELO_médio_liga                    [1.0 = média da liga]
Z_adj        = Z_bruto / SOS_norm                          [tabela fácil amplifica]

confiança    = min(N_jogos / N_REF, 1.0)
alpha_ef     = ALPHA × confiança
Gols_adj     = alpha_ef × Gols_N10 + (1 - alpha_ef) × xG_N10

Custo_Casa   = Gols_adj_casa / (1/odd_casa)               [maior = melhor]
Valor_Casa   = Gols_adj_casa × (1/odd_fora)               [maior = melhor]
Luck         = Gols_N10 − xG_N10                           [próximo 0 = saudável]
Delta        = prob_modelo − prob_mercado                   [>0 = modelo vê valor]

IL           = 0.25×EV + 0.20×SOS + 0.20×Z + 0.20×Custo + 0.15×Luck
```

## Classificação final

| Classificação | Condição |
|---|---|
| `VALOR_REAL_CASA/FORA` | IL > 0.65 + EV > 3% |
| `FALSO_FAVORITO_CASA/FORA` | prob_mercado > 55% + IL < 35% |
| `POTENCIAL_CASA/FORA` | Delta > 8% + IL > 50% |
| `NEUTRO` | Demais casos |


In [1]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 0 — Imports e Constantes
# Único ponto de configuração de todo o pipeline
# ═══════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
from IPython.display import display

# ── Caminhos ──────────────────────────────────────────────
PATH_RODADA  = 'rodada_atual_v3.csv'      # entrada: rodada atual sem SOS/IL
PATH_BASE    = 'base_historica_v3.csv'    # entrada: base histórica com ELO
PATH_OUTPUT  = 'rodada_atual_sos_v3.csv'  # saída: CSV completo com todas as métricas

# ── Parâmetros SOS ────────────────────────────────────────
N_JOGOS = 10      # últimos N jogos para calcular SOS
DECAY   = 0.85    # decay exponencial: jogo[0]=1.0, jogo[1]=0.85, jogo[2]=0.72...

# ── Parâmetros RTM — μ/σ calibrados na base histórica (n≈5k jogos) ──
RTM_PARAMS = {
    'gols_home': {'mu': 1.5336, 'sigma': 0.6711, 'col': 'gols_marc_home'},
    'gols_away': {'mu': 1.2656, 'sigma': 0.5790, 'col': 'gols_marc_away'},
    'xg_home':   {'mu': 1.5592, 'sigma': 0.3601, 'col': 'xg_home_n10'},
    'xg_away':   {'mu': 1.3208, 'sigma': 0.3242, 'col': 'xg_away_n10'},
    'vit_home':  {'mu': 0.5002, 'sigma': 0.2456, 'col': 'pct_vit_home'},
    'vit_away':  {'mu': 0.3916, 'sigma': 0.2172, 'col': 'pct_vit_away'},
}

# ── Parâmetros IL ─────────────────────────────────────────
ALPHA_SHRINKAGE = 0.60   # peso gols reais vs xG âncora (N=10: alpha_ef=0.60, N=5: 0.30)
N_REF           = 10     # N de referência para confiança amostral
DELTA_THRESHOLD = 0.08   # divergência mínima modelo vs mercado para classificar Potencial

PESOS_IL = {
    'ev':    0.25,   # EV positivo    — mercado subestima
    'sos':   0.20,   # SOS alto       — forma veio de tabela difícil
    'z':     0.20,   # Z sustentável  — não está em pico atípico
    'custo': 0.20,   # Custo do Gol   — eficiência real contra expectativa
    'luck':  0.15,   # Luck baixo     — não depende de sorte na conversão
}

# ── Limiares de classificação IL ─────────────────────────
IL_VALOR_MIN   = 0.65   # IL acima = candidato a Valor Real
IL_FALSO_MAX   = 0.35   # IL abaixo + mercado favorece = Falso Favorito
PROB_MERC_MIN  = 0.55   # prob mercado acima = time considerado favorito
EV_MIN_IL      = 0.03   # EV mínimo para classificar Valor Real

print('✅ Constantes configuradas')
print(f'   Entrada : {PATH_RODADA} + {PATH_BASE}')
print(f'   Saída   : {PATH_OUTPUT}')
print(f'   SOS     : N={N_JOGOS} | decay={DECAY}')
print(f'   IL      : alpha={ALPHA_SHRINKAGE} | delta_th={DELTA_THRESHOLD}')
print(f'   Pesos IL: {PESOS_IL}')

✅ Constantes configuradas
   Entrada : rodada_atual_v3.csv + base_historica_v3.csv
   Saída   : rodada_atual_sos_v3.csv
   SOS     : N=10 | decay=0.85
   IL      : alpha=0.6 | delta_th=0.08
   Pesos IL: {'ev': 0.25, 'sos': 0.2, 'z': 0.2, 'custo': 0.2, 'luck': 0.15}


In [2]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 1 — Carregar CSVs
# ═══════════════════════════════════════════════════════════
ra = pd.read_csv(PATH_RODADA)
bh = pd.read_csv(PATH_BASE)

print(f'rodada_atual  : {ra.shape[0]} jogos | {ra.shape[1]} colunas')
print(f'base_historica: {bh.shape[0]} jogos | {bh.shape[1]} colunas')
print(f'Ligas rodada  : {ra["league"].nunique()} | Ligas base: {bh["league"].nunique()}')
print()

# Cobertura de IDs entre rodada e base
ids_ra = set(ra['home_id'].tolist() + ra['away_id'].tolist())
ids_bh = set(bh['home_id'].dropna().tolist()) | set(bh['away_id'].dropna().tolist())
cobertura = len(ids_ra & ids_bh) / len(ids_ra) * 100
print(f'Cobertura de times por ID: {cobertura:.0f}%')

# Avisar se colunas SOS/IL já existem (serão recalculadas)
cols_existentes = [c for c in ra.columns if 'sos' in c.lower() or c.startswith('z_')
                   or c.startswith('IL') or 'custo_gol' in c or 'luck_' in c]
if cols_existentes:
    print(f'⚠️  {len(cols_existentes)} colunas existentes serão recalculadas: {cols_existentes[:5]}...')
else:
    print('ℹ️  Calculando do zero')

display(ra[['league','home_team','away_team','elo_home','elo_away',
            'gols_marc_home','xg_home_n10','n10_home']].head(5))

rodada_atual  : 225 jogos | 62 colunas
base_historica: 6027 jogos | 78 colunas
Ligas rodada  : 26 | Ligas base: 31

Cobertura de times por ID: 92%
ℹ️  Calculando do zero


,league,home_team,away_team,elo_home,elo_away,gols_marc_home,xg_home_n10,n10_home
0,Alemanha_1 Bundesliga,Eintracht Frankfurt,Köln,1495.107608,1421.346544,1.5,1.186,10
1,Alemanha_1 Bundesliga,Freiburg,Bayern München,1503.234858,1693.601163,1.2,1.387,10
2,Alemanha_1 Bundesliga,Borussia M'gladbach,Heidenheim,1463.793968,1353.600950,1.0,1.218,10
3,Alemanha_1 Bundesliga,Hamburger SV,Augsburg,1468.770849,1459.232899,1.4,1.105,10
4,Alemanha_1 Bundesliga,Stuttgart,Borussia Dortmund,1600.112814,1648.723114,2.4,1.688,10


In [3]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 2 — Funções (SOS · Z-Score · IL)
# ═══════════════════════════════════════════════════════════

# ── SOS ──────────────────────────────────────────────────
def calc_sos(team_id, liga, bh_df, elo_liga_map, n=N_JOGOS, decay=DECAY):
    """
    SOS_raw  = média ponderada do ELO dos adversários (decay exponencial)
    SOS_norm = SOS_raw / ELO_médio_liga  →  1.0 = média da liga
    """
    j_casa = bh_df[bh_df['home_id'] == team_id][['date_unix','elo_away']].copy()
    j_casa.columns = ['date_unix','opp_elo']
    j_fora = bh_df[bh_df['away_id'] == team_id][['date_unix','elo_home']].copy()
    j_fora.columns = ['date_unix','opp_elo']

    todos = (
        pd.concat([j_casa, j_fora])
        .dropna(subset=['opp_elo'])
        .query('opp_elo > 1000')
        .sort_values('date_unix', ascending=False)
        .head(n)
        .reset_index(drop=True)
    )
    if len(todos) == 0:
        return None, None, 0

    pesos    = np.array([decay**i for i in range(len(todos))])
    sos_raw  = float(np.average(todos['opp_elo'], weights=pesos))
    elo_med  = elo_liga_map.get(liga, 1500.0)
    sos_norm = sos_raw / elo_med if elo_med > 0 else 1.0
    return round(sos_raw, 2), round(sos_norm, 4), len(todos)


# ── Z-Score ──────────────────────────────────────────────
def z_score(val, mu, sigma):
    """Z = (x − μ) / σ"""
    return round((val - mu) / sigma, 4) if sigma > 0 else 0.0

def z_ajustado(z_bruto, sos_norm):
    """Z_adj = Z_bruto / SOS_norm  →  tabela fácil amplifica o alerta"""
    sos = sos_norm if sos_norm and sos_norm > 0 else 1.0
    return round(z_bruto / sos, 4)


# ── IL ───────────────────────────────────────────────────
def odd_para_prob(serie_odd):
    """prob = 1 / odd  (retorna NaN para odds inválidas)"""
    o = pd.to_numeric(serie_odd, errors='coerce')
    return np.where(o > 1.0, 1.0 / o, np.nan)

def calcular_gols_ajustados(gols, xg, n, alpha=ALPHA_SHRINKAGE, n_ref=N_REF):
    """
    Gols_adj = alpha_ef × Gols_N10 + (1 − alpha_ef) × xG_N10
    alpha_ef = alpha × min(N/N_ref, 1.0)  →  N pequeno ancora mais no xG
    """
    confianca = np.minimum(n / n_ref, 1.0)
    alpha_ef  = alpha * confianca
    return alpha_ef * gols + (1.0 - alpha_ef) * xg

def normalizar_01(serie, q_low=0.05, q_high=0.95):
    """Normaliza para [0,1] usando percentis — robusto a outliers"""
    lo = serie.quantile(q_low)
    hi = serie.quantile(q_high)
    if hi == lo:
        return pd.Series(0.5, index=serie.index)
    return ((serie - lo) / (hi - lo)).clip(0, 1)

def classificar_jogo(row):
    """
    Hierarquia: FALSO_FAVORITO > VALOR_REAL > POTENCIAL > NEUTRO
    """
    il_h  = row['IL_casa']
    il_a  = row['IL_fora']
    pm_h  = row.get('prob_mercado_casa') or 0
    pm_a  = row.get('prob_mercado_fora') or 0
    ev_h  = row['ev_bc']  if pd.notna(row.get('ev_bc'))  else 0
    ev_a  = row['ev_bf']  if pd.notna(row.get('ev_bf'))  else 0
    dlt_h = row.get('delta_bc') or 0
    dlt_a = row.get('delta_bf') or 0

    if pm_h > PROB_MERC_MIN and il_h < IL_FALSO_MAX:
        return 'FALSO_FAVORITO_CASA'
    if pm_a > PROB_MERC_MIN and il_a < IL_FALSO_MAX:
        return 'FALSO_FAVORITO_FORA'
    if il_h > IL_VALOR_MIN and ev_h > EV_MIN_IL:
        return 'VALOR_REAL_CASA'
    if il_a > IL_VALOR_MIN and ev_a > EV_MIN_IL:
        return 'VALOR_REAL_FORA'
    if dlt_h > DELTA_THRESHOLD and il_h > 0.50:
        return 'POTENCIAL_CASA'
    if dlt_a > DELTA_THRESHOLD and il_a > 0.50:
        return 'POTENCIAL_FORA'
    return 'NEUTRO'


print('✅ Funções definidas: calc_sos | z_score | z_ajustado | odd_para_prob')
print('                     calcular_gols_ajustados | normalizar_01 | classificar_jogo')

✅ Funções definidas: calc_sos | z_score | z_ajustado | odd_para_prob
                     calcular_gols_ajustados | normalizar_01 | classificar_jogo


In [4]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 3 — ELO médio por liga + SOS
# ═══════════════════════════════════════════════════════════
import time

# ELO médio por liga (base de normalização do SOS)
elo_liga = {}
for liga in bh['league'].unique():
    sub  = bh[bh['league'] == liga]
    elos = pd.concat([sub['elo_home'], sub['elo_away']]).dropna()
    er   = elos[elos != 1500.0]  # excluir ELO padrão inicial
    elo_liga[liga] = float(er.mean()) if len(er) > 10 else 1500.0

print(f'ELO médio calculado para {len(elo_liga)} ligas')
print()

# Calcular SOS para todos os times da rodada
print(f'Calculando SOS para {len(ra)} jogos ({len(ra)*2} times)...')
sh_r, sh_n, sh_c = [], [], []
sa_r, sa_n, sa_c = [], [], []

for _, row in ra.iterrows():
    r, n, c = calc_sos(row['home_id'], row['league'], bh, elo_liga)
    sh_r.append(r); sh_n.append(n); sh_c.append(c)
    r, n, c = calc_sos(row['away_id'], row['league'], bh, elo_liga)
    sa_r.append(r); sa_n.append(n); sa_c.append(c)

for col in ['sos_home_raw','sos_home_norm','sos_home_n',
            'sos_away_raw','sos_away_norm','sos_away_n']:
    if col in ra.columns: ra.drop(columns=[col], inplace=True)

ra['sos_home_raw']  = sh_r
ra['sos_home_norm'] = sh_n
ra['sos_home_n']    = sh_c
ra['sos_away_raw']  = sa_r
ra['sos_away_norm'] = sa_n
ra['sos_away_n']    = sa_c

print(f'✅ SOS calculado')
print(f'   home: min={ra["sos_home_norm"].min():.4f} max={ra["sos_home_norm"].max():.4f} μ={ra["sos_home_norm"].mean():.4f}')
print(f'   away: min={ra["sos_away_norm"].min():.4f} max={ra["sos_away_norm"].max():.4f} μ={ra["sos_away_norm"].mean():.4f}')

display(ra[['league','home_team','away_team',
            'sos_home_norm','sos_home_n',
            'sos_away_norm','sos_away_n']].head(6))

ELO médio calculado para 31 ligas

Calculando SOS para 225 jogos (450 times)...
✅ SOS calculado
   home: min=0.9735 max=1.0316 μ=1.0000
   away: min=0.9680 max=1.0285 μ=1.0001


,league,home_team,away_team,sos_home_norm,sos_home_n,sos_away_norm,sos_away_n
0,Alemanha_1 Bundesliga,Eintracht Frankfurt,Köln,0.9910,10,1.0122,10
1,Alemanha_1 Bundesliga,Freiburg,Bayern München,0.9890,10,1.0078,10
2,Alemanha_1 Bundesliga,Borussia M'gladbach,Heidenheim,1.0012,10,1.0141,10
3,Alemanha_1 Bundesliga,Hamburger SV,Augsburg,1.0007,10,1.0147,10
4,Alemanha_1 Bundesliga,Stuttgart,Borussia Dortmund,0.9776,10,0.9924,10
5,Alemanha_1 Bundesliga,Bayer Leverkusen,Wolfsburg,0.9894,10,1.0007,10


In [5]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 4 — Z-Scores brutos + ajustados por SOS
# ═══════════════════════════════════════════════════════════

for key, p in RTM_PARAMS.items():
    col_z    = f'z_{key}'
    col_zadj = f'z_{key}_adj'

    if p['col'] not in ra.columns:
        print(f'⚠️  Coluna {p["col"]} não encontrada — pulando {key}')
        continue

    for c in [col_z, col_zadj]:
        if c in ra.columns: ra.drop(columns=[c], inplace=True)

    # Z bruto
    ra[col_z] = ra[p['col']].apply(
        lambda x: z_score(float(x) if pd.notna(x) else p['mu'], p['mu'], p['sigma'])
    )

    # Z ajustado: home → sos_home_norm | away → sos_away_norm
    sos_col = 'sos_home_norm' if 'home' in key else 'sos_away_norm'
    ra[col_zadj] = ra.apply(lambda r: z_ajustado(r[col_z], r[sos_col]), axis=1)

z_cols = [c for c in ra.columns if c.startswith('z_')]
print(f'✅ Z-Scores calculados: {z_cols}')
print()
print('Amostra Z bruto vs Z_adj:')
display(ra[['home_team','away_team',
            'z_gols_home','sos_home_norm','z_gols_home_adj',
            'z_gols_away','sos_away_norm','z_gols_away_adj']].head(6))

✅ Z-Scores calculados: ['z_gols_home', 'z_gols_home_adj', 'z_gols_away', 'z_gols_away_adj', 'z_xg_home', 'z_xg_home_adj', 'z_xg_away', 'z_xg_away_adj', 'z_vit_home', 'z_vit_home_adj', 'z_vit_away', 'z_vit_away_adj']

Amostra Z bruto vs Z_adj:


,home_team,away_team,z_gols_home,sos_home_norm,z_gols_home_adj,z_gols_away,sos_away_norm,z_gols_away_adj
0,Eintracht Frankfurt,Köln,-0.0501,0.9910,-0.0506,0.0594,1.0122,0.0587
1,Freiburg,Bayern München,-0.4971,0.9890,-0.5026,3.1682,1.0078,3.1437
2,Borussia M'gladbach,Heidenheim,-0.7951,1.0012,-0.7941,-0.2860,1.0141,-0.2820
3,Hamburger SV,Augsburg,-0.1991,1.0007,-0.1990,0.4048,1.0147,0.3989
4,Stuttgart,Borussia Dortmund,1.2910,0.9776,1.3206,2.3047,0.9924,2.3223
5,Bayer Leverkusen,Wolfsburg,0.3970,0.9894,0.4013,-0.6314,1.0007,-0.6310


In [6]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 5 — Custo/Valor do Gol · Luck Factor · Delta
# ═══════════════════════════════════════════════════════════

# Odds: preferir atual se existir, fallback para D-2
# odd_home_atual só existe após o scheduler — não obrigatória aqui
def get_odd_col(df, col_atual, col_d2):
    if col_atual in df.columns:
        atual = pd.to_numeric(df[col_atual], errors='coerce')
        return atual.where(atual > 1.0, pd.to_numeric(df[col_d2], errors='coerce'))
    return pd.to_numeric(df[col_d2], errors='coerce')

odd_home = get_odd_col(ra, 'odd_home_atual', 'odd_home_d2')
odd_away = get_odd_col(ra, 'odd_away_atual', 'odd_away_d2')

prob_casa = odd_para_prob(odd_home)
prob_fora = odd_para_prob(odd_away)

ra['prob_mercado_casa'] = prob_casa.round(4)
ra['prob_mercado_fora'] = prob_fora.round(4)

# Gols ajustados (xG âncora + peso amostral)
gols_h_adj = calcular_gols_ajustados(ra['gols_marc_home'], ra['xg_home_n10'], ra['n10_home'])
gols_a_adj = calcular_gols_ajustados(ra['gols_marc_away'], ra['xg_away_n10'], ra['n10_away'])
ra['gols_home_adj'] = gols_h_adj.round(3)
ra['gols_away_adj'] = gols_a_adj.round(3)

# Custo do Gol  → Gols_adj / prob_própria   (maior = melhor)
# Valor do Gol  → Gols_adj × prob_adversário (maior = melhor)
ra['custo_gol_casa'] = (gols_h_adj / prob_casa).round(4)
ra['custo_gol_fora'] = (gols_a_adj / prob_fora).round(4)
ra['valor_gol_casa'] = (gols_h_adj * prob_fora).round(4)
ra['valor_gol_fora'] = (gols_a_adj * prob_casa).round(4)

# Luck Factor  →  ΔxG = Gols − xG  (próximo de 0 = saudável)
ra['luck_home'] = (ra['gols_marc_home'] - ra['xg_home_n10']).round(3)
ra['luck_away'] = (ra['gols_marc_away'] - ra['xg_away_n10']).round(3)

# Delta  →  prob_modelo − prob_mercado  (>0 = modelo vê mais valor)
ra['delta_bc'] = (ra['prob_bc'] - prob_casa).round(4)
ra['delta_bf'] = (ra['prob_bf'] - prob_fora).round(4)

print('✅ Custo/Valor do Gol, Luck Factor e Delta calculados')
display(ra[['home_team','away_team',
            'gols_marc_home','gols_home_adj','xg_home_n10',
            'custo_gol_casa','valor_gol_casa',
            'luck_home','delta_bc']].head(6))

✅ Custo/Valor do Gol, Luck Factor e Delta calculados


,home_team,away_team,gols_marc_home,gols_home_adj,xg_home_n10,custo_gol_casa,valor_gol_casa,luck_home,delta_bc
0,Eintracht Frankfurt,Köln,1.5,1.374,1.186,NaN,NaN,0.314,NaN
1,Freiburg,Bayern München,1.2,1.275,1.387,10.3259,0.9106,-0.187,0.0138
2,Borussia M'gladbach,Heidenheim,1.0,1.087,1.218,1.7721,0.1878,-0.218,-0.0520
3,Hamburger SV,Augsburg,1.4,1.282,1.105,2.8717,0.3760,0.295,-0.0259
4,Stuttgart,Borussia Dortmund,2.4,2.115,1.688,4.7169,0.6845,0.712,-0.0697
5,Bayer Leverkusen,Wolfsburg,1.8,1.727,1.617,2.6075,0.2698,0.183,0.0575


In [7]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 6 — Índice de Legitimidade (IL) + Classificação
# ═══════════════════════════════════════════════════════════

# Componentes casa (cada um normalizado em [0,1])
c1_h = normalizar_01(ra['ev_bc'].fillna(0))               # EV+
c2_h = normalizar_01(ra['sos_home_norm'])                  # SOS alto
c3_h = normalizar_01(-ra['z_gols_home_adj'].abs())         # Z próximo de 0
c4_h = normalizar_01(ra['custo_gol_casa'])                 # Custo alto
c5_h = normalizar_01(-ra['luck_home'].abs())               # Luck próximo de 0

ra['IL_casa'] = (
    PESOS_IL['ev']    * c1_h +
    PESOS_IL['sos']   * c2_h +
    PESOS_IL['z']     * c3_h +
    PESOS_IL['custo'] * c4_h +
    PESOS_IL['luck']  * c5_h
).round(4)

# Componentes fora
c1_a = normalizar_01(ra['ev_bf'].fillna(0))
c2_a = normalizar_01(ra['sos_away_norm'])
c3_a = normalizar_01(-ra['z_gols_away_adj'].abs())
c4_a = normalizar_01(ra['custo_gol_fora'])
c5_a = normalizar_01(-ra['luck_away'].abs())

ra['IL_fora'] = (
    PESOS_IL['ev']    * c1_a +
    PESOS_IL['sos']   * c2_a +
    PESOS_IL['z']     * c3_a +
    PESOS_IL['custo'] * c4_a +
    PESOS_IL['luck']  * c5_a
).round(4)

# Classificação
ra['classificacao_IL'] = ra.apply(classificar_jogo, axis=1)

print('✅ IL calculado')
print(f'   IL_casa: min={ra["IL_casa"].min():.3f} max={ra["IL_casa"].max():.3f} μ={ra["IL_casa"].mean():.3f}')
print(f'   IL_fora: min={ra["IL_fora"].min():.3f} max={ra["IL_fora"].max():.3f} μ={ra["IL_fora"].mean():.3f}')
print()
print('Distribuição das classificações:')
print(ra['classificacao_IL'].value_counts())

✅ IL calculado
   IL_casa: min=0.108 max=0.888 μ=0.534
   IL_fora: min=0.109 max=0.870 μ=0.543

Distribuição das classificações:
classificacao_IL
NEUTRO                 183
VALOR_REAL_FORA         17
VALOR_REAL_CASA         11
FALSO_FAVORITO_CASA      5
FALSO_FAVORITO_FORA      3
POTENCIAL_FORA           3
POTENCIAL_CASA           3
Name: count, dtype: int64


In [8]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 7 — Análise por classificação
# ═══════════════════════════════════════════════════════════

cols_casa = ['league','home_team','away_team',
             'IL_casa','custo_gol_casa','valor_gol_casa',
             'luck_home','delta_bc','sos_home_norm','z_gols_home_adj',
             'ev_bc','prob_bc','prob_mercado_casa']
cols_fora = ['league','home_team','away_team',
             'IL_fora','custo_gol_fora','valor_gol_fora',
             'luck_away','delta_bf','sos_away_norm','z_gols_away_adj',
             'ev_bf','prob_bf','prob_mercado_fora']

for cl, cols in [
    ('VALOR_REAL_CASA',      cols_casa),
    ('VALOR_REAL_FORA',      cols_fora),
    ('FALSO_FAVORITO_CASA',  cols_casa),
    ('FALSO_FAVORITO_FORA',  cols_fora),
    ('POTENCIAL_CASA',       cols_casa),
    ('POTENCIAL_FORA',       cols_fora),
]:
    sub = ra[ra['classificacao_IL'] == cl]
    if len(sub) == 0: continue
    col_il = 'IL_casa' if 'CASA' in cl else 'IL_fora'
    print(f'\n=== {cl} ({len(sub)} jogos) ===')
    display(sub.sort_values(col_il, ascending='FALSO' in cl)[cols])


=== VALOR_REAL_CASA (11 jogos) ===


,league,home_team,away_team,IL_casa,custo_gol_casa,valor_gol_casa,luck_home,delta_bc,sos_home_norm,z_gols_home_adj,ev_bc,prob_bc,prob_mercado_casa
191,Colombia_1 Liga BetPlay,Deportivo Pasto,Atlético Nacional,0.8880,5.1758,0.7214,0.053,0.0617,0.9983,-0.0502,0.215863,0.347389,0.2857
64,Espanha_2 Segunda Division,Real Sociedad II,SD Eibar,0.7746,5.4042,0.7454,0.692,0.0837,1.0144,0.9789,0.235244,0.439589,0.3559
102,Holanda_2 Eerste Divisie,Eindhoven,Emmen,0.7731,4.1109,0.6705,-0.392,0.0631,0.9970,0.0992,0.147636,0.490443,0.4274
27,Australia_1 A-League,Adelaide United,Auckland,0.7622,7.2024,1.0151,0.276,0.1226,0.9956,0.8477,0.443918,0.398872,0.2762
25,Australia_1 A-League,Central Coast Mariners,Perth Glory FC,0.7617,3.0406,0.4367,0.410,0.0418,1.0106,-0.1970,0.102803,0.448294,0.4065
41,Escocia_1 Premiership,Motherwell,Falkirk,0.7544,3.4267,0.4672,0.043,0.0506,0.9998,0.5461,0.092114,0.600062,0.5495
162,Brasileiro_A Serie A,Fluminense,Corinthians,0.7325,3.2182,0.3926,0.126,0.0465,0.9987,0.3975,0.087389,0.578399,0.5319
68,Espanha_2 Segunda Division,Ceuta,Cádiz,0.6975,2.8943,0.4313,-0.039,0.0322,0.9936,-0.3503,0.070876,0.486762,0.4545
72,Espanha_2 Segunda Division,SD Eibar,UD Las Palmas,0.6948,2.8870,0.4080,-0.111,0.0307,0.9981,-0.4980,0.071138,0.461697,0.4310
28,Australia_1 A-League,Brisbane Roar,Sydney FC,0.6822,4.5893,0.6173,-0.491,0.0427,0.9876,-0.6542,0.151214,0.325202,0.2825



=== VALOR_REAL_FORA (17 jogos) ===


,league,home_team,away_team,IL_fora,custo_gol_fora,valor_gol_fora,luck_away,delta_bf,sos_away_norm,z_gols_away_adj,ev_bf,prob_bf,prob_mercado_fora
160,Brasileiro_A Serie A,Palmeiras,Grêmio,0.8696,9.0207,0.8884,0.170,0.0676,1.0093,0.4011,0.451722,0.217324,0.1497
182,Chile_1 Primera Division,Coquimbo Unido,Cobresal,0.8565,10.0543,0.9698,0.039,0.0684,0.9971,0.2823,0.488957,0.208246,0.1399
189,Colombia_1 Liga BetPlay,Millonarios,Fortaleza CEIF,0.8487,7.4220,0.7849,-0.022,0.0574,1.0087,-0.1123,0.352675,0.220305,0.1629
199,Colombia_1 Liga BetPlay,Atlético Nacional,Cúcuta Deportivo,0.7923,18.1367,1.1058,0.238,0.0566,0.9897,0.2345,0.787141,0.128571,0.0719
224,Brasileiro_B Serie B,Criciúma,Athletic Club,0.7738,9.7728,1.1150,0.040,0.1667,1.0000,1.2684,0.830249,0.367520,0.2008
82,Franca_1 Ligue 1,Strasbourg,Nice,0.7628,6.6368,0.7172,-0.084,0.0146,1.0165,-0.1115,0.078645,0.200492,0.1859
187,Colombia_1 Liga BetPlay,Santa Fe,Llaneros,0.7359,5.5171,0.6441,-0.173,0.0801,1.0033,-0.4572,0.413288,0.273893,0.1938
98,Holanda_1 Eredivisie,PSV,Utrecht,0.7315,10.4664,1.0245,-0.310,0.0131,0.9997,0.0594,0.096097,0.149129,0.1361
2,Alemanha_1 Bundesliga,Borussia M'gladbach,Heidenheim,0.7313,6.4524,0.6837,-0.036,0.0087,1.0141,-0.2820,0.050234,0.181388,0.1727
84,Franca_1 Ligue 1,PSG,Toulouse,0.7236,16.3849,1.1218,-0.258,0.0175,0.9938,0.2335,0.191224,0.109287,0.0917



=== FALSO_FAVORITO_CASA (5 jogos) ===


...

,league,home_team,away_team,IL_fora,custo_gol_fora,valor_gol_fora,luck_away,delta_bf,sos_away_norm,z_gols_away_adj,ev_bf,prob_bf,prob_mercado_fora
1,Alemanha_1 Bundesliga,Freiburg,Bayern München,0.2671,3.8573,0.3401,0.862,-0.0588,1.0078,3.1437,-0.082253,0.655533,0.7143
128,Portugal_1 Primeira Liga,Moreirense FC,Sporting Braga,0.2880,3.1553,0.3235,1.026,-0.0825,1.0285,2.0729,-0.124514,0.579792,0.6623
176,Argentina_1 Liga Profesional,Aldosivi,Argentinos Juniors,0.3040,2.1512,0.2397,-1.208,-0.0509,1.0035,-1.0308,-0.089565,0.517293,0.5682



=== POTENCIAL_CASA (3 jogos) ===


,league,home_team,away_team,IL_casa,custo_gol_casa,valor_gol_casa,luck_home,delta_bc,sos_home_norm,z_gols_home_adj,ev_bc,prob_bc,prob_mercado_casa
149,Brasileiro_A Serie A,Vasco da Gama,Botafogo,0.6477,4.4934,0.6006,-1.633,0.1604,1.0044,-1.2861,0.359341,0.606849,0.4464
150,Brasileiro_A Serie A,Bragantino,Flamengo,0.6390,4.4489,0.5721,-0.788,0.0877,1.0059,-1.5312,0.355176,0.334611,0.2469
121,Italia_1 Serie A,Hellas Verona,Fiorentina,0.6361,3.8506,0.4587,-0.213,0.0960,0.9848,-1.1100,0.417593,0.325884,0.2299



=== POTENCIAL_FORA (3 jogos) ===


,league,home_team,away_team,IL_fora,custo_gol_fora,valor_gol_fora,luck_away,delta_bf,sos_away_norm,z_gols_away_adj,ev_bf,prob_bf,prob_mercado_fora
188,Colombia_1 Liga BetPlay,Alianza Petrolera,Deportivo Pasto,0.6394,3.8745,0.5377,0.053,0.1000,0.9983,0.4055,0.262030,0.481691,0.3817
91,Franca_2 Ligue 2,Dunkerque,Rodez,0.6185,5.2694,0.7166,0.346,0.1042,0.9890,0.4093,0.403190,0.362581,0.2584
158,Brasileiro_A Serie A,Coritiba,Vasco da Gama,0.5812,6.5195,0.8500,-1.633,0.1183,1.0044,-1.0299,0.384591,0.426028,0.3077


In [9]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 8 — Exportar CSV final
# ═══════════════════════════════════════════════════════════

ra.to_csv(PATH_OUTPUT, index=False)

todas_novas = [
    # SOS
    'sos_home_raw','sos_home_norm','sos_home_n',
    'sos_away_raw','sos_away_norm','sos_away_n',
    # Z-Scores
    'z_gols_home','z_gols_home_adj','z_gols_away','z_gols_away_adj',
    'z_xg_home','z_xg_home_adj','z_xg_away','z_xg_away_adj',
    'z_vit_home','z_vit_home_adj','z_vit_away','z_vit_away_adj',
    # IL
    'prob_mercado_casa','prob_mercado_fora',
    'gols_home_adj','gols_away_adj',
    'custo_gol_casa','custo_gol_fora',
    'valor_gol_casa','valor_gol_fora',
    'luck_home','luck_away',
    'delta_bc','delta_bf',
    'IL_casa','IL_fora','classificacao_IL',
]

print(f'✅ Exportado: {PATH_OUTPUT}')
print(f'   Shape final: {ra.shape}  (+{len(todas_novas)} colunas novas)')
print()
print('Colunas geradas neste pipeline:')
grupos = [
    ('SOS (6)',     [c for c in todas_novas if 'sos' in c]),
    ('Z-Score (12)',[c for c in todas_novas if c.startswith('z_')]),
    ('IL (15)',     [c for c in todas_novas if c not in
                    [c2 for c2 in todas_novas if 'sos' in c2 or c2.startswith('z_')]]),
]
for nome, cols in grupos:
    presentes = [c for c in cols if c in ra.columns]
    print(f'  {nome}: {presentes}')

print()
print('Distribuição final das classificações:')
print(ra['classificacao_IL'].value_counts())
print()
print('📌 Próximo passo: envie rodada_atual_sos_v3.csv para o Claude')
print('   → gerar index.html com SOS+RTM+IL embutido no painel.')
display(ra[['home_team','away_team','IL_casa','IL_fora','classificacao_IL']].head(8))

✅ Exportado: rodada_atual_sos_v3.csv
   Shape final: (225, 95)  (+33 colunas novas)

Colunas geradas neste pipeline:
  SOS (6): ['sos_home_raw', 'sos_home_norm', 'sos_home_n', 'sos_away_raw', 'sos_away_norm', 'sos_away_n']
  Z-Score (12): ['z_gols_home', 'z_gols_home_adj', 'z_gols_away', 'z_gols_away_adj', 'z_xg_home', 'z_xg_home_adj', 'z_xg_away', 'z_xg_away_adj', 'z_vit_home', 'z_vit_home_adj', 'z_vit_away', 'z_vit_away_adj']
  IL (15): ['prob_mercado_casa', 'prob_mercado_fora', 'gols_home_adj', 'gols_away_adj', 'custo_gol_casa', 'custo_gol_fora', 'valor_gol_casa', 'valor_gol_fora', 'luck_home', 'luck_away', 'delta_bc', 'delta_bf', 'IL_casa', 'IL_fora', 'classificacao_IL']

Distribuição final das classificações:
classificacao_IL
NEUTRO                 183
VALOR_REAL_FORA         17
VALOR_REAL_CASA         11
FALSO_FAVORITO_CASA      5
FALSO_FAVORITO_FORA      3
POTENCIAL_FORA           3
POTENCIAL_CASA           3
Name: count, dtype: int64

📌 Próximo passo: envie rodada_atual_sos_v3.

,home_team,away_team,IL_casa,IL_fora,classificacao_IL
0,Eintracht Frankfurt,Köln,NaN,NaN,NEUTRO
1,Freiburg,Bayern München,0.7678,0.2671,FALSO_FAVORITO_FORA
2,Borussia M'gladbach,Heidenheim,0.5082,0.7313,VALOR_REAL_FORA
3,Hamburger SV,Augsburg,0.6218,0.6943,VALOR_REAL_FORA
4,Stuttgart,Borussia Dortmund,0.3778,0.3300,NEUTRO
5,Bayer Leverkusen,Wolfsburg,0.6388,0.4636,NEUTRO
6,Union Berlin,St. Pauli,NaN,NaN,NEUTRO
7,Werder Bremen,RB Leipzig,0.3438,0.5209,NEUTRO
